# 05 — Dashboard

Combined view. Assumes notebooks 02 and 03 have already been run so the analyses table has sentiment + topics. This notebook is read-only — it doesn't run new analyses.

In [1]:
import sys, json
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

from journal.config import LANCE_ROOT
from journal.store import Store

store = Store(LANCE_ROOT)
e = store.entries_to_pandas()
a = store.analyses_to_pandas()
print(f"{len(e):,} entries, {len(a):,} analyses")

3,035 entries, 400 analyses


In [2]:
s = a[(a["question_id"]=="sentiment") & a["parsed_ok"]].copy()
s["level"] = s["result_json"].apply(lambda x: json.loads(x)["level"])
s["score"] = s["result_json"].apply(lambda x: json.loads(x)["score"])
df = s.merge(e[["id","date","day_of_week","time_of_day"]], left_on="entry_id", right_on="id")
df["date"] = pd.to_datetime(df["date"])

order = ["extreme_negative","negative","neutral","positive","extreme_positive"]
day_order = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
tod_order = ["morning","afternoon","evening","night"]

ct = df.groupby(["day_of_week","level"]).size().reset_index(name="n")
ct["day_of_week"] = pd.Categorical(ct["day_of_week"], categories=day_order, ordered=True)
px.bar(ct, x="day_of_week", y="n", color="level", category_orders={"level": order},
       title="Sentiment by day of week").show()

ct = df.groupby(["time_of_day","level"]).size().reset_index(name="n")
ct["time_of_day"] = pd.Categorical(ct["time_of_day"], categories=tod_order, ordered=True)
px.bar(ct, x="time_of_day", y="n", color="level", category_orders={"level": order},
       title="Sentiment by time of day").show()

wk = df.set_index("date")["score"].resample("W").mean().reset_index()
px.line(wk, x="date", y="score", title="Weekly average sentiment score").show()

In [3]:
t = a[(a["question_id"]=="topics") & a["parsed_ok"]].copy()
t["topics"] = t["result_json"].apply(lambda x: json.loads(x)["topics"])
exploded = t.explode("topics")
top = exploded["topics"].value_counts().head(25).reset_index()
top.columns = ["topic","n"]
fig = px.bar(top, x="n", y="topic", orientation="h", title="Top 25 topics")
fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()

In [4]:
activity = e.copy()
activity["date"] = pd.to_datetime(activity["date"])
per_week = activity.set_index("date").resample("W").size().reset_index(name="entries")
px.line(per_week, x="date", y="entries", title="Entries per week").show()